# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
dupes_count = df.duplicated().sum()
df = df.drop_duplicates().copy()
log('duplicates', 'dropped exact duplicate rows', dupes_count)

[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)
assert df['price'].dtype == float
log('price', 'stripped dollar signs and converted price column to float', len(df))

[price] stripped dollar signs and converted price column to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

invalid_mask = df['qty'].isna() | (df['qty'] <= 0)
invalid_count = invalid_mask.sum()

df = df[~invalid_mask].copy()
log('qty_filter', 'dropped rows with missing or negative quantities', invalid_count)

[qty_filter] dropped rows with missing or negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print("Before canonicalization:")
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].map(ITEM_MAP)
log('item', 'canonicalized item spellings into 3 standard products', len(df))

print("\nAfter canonicalization:")
print(df['item'].value_counts())

Before canonicalization:
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] canonicalized item spellings into 3 standard products (275 row(s))

After canonicalization:
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print("Before normalization:")
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
    'Merch': 'Merch',
    'Apparel': 'Apparel'
}

df['category'] = df['category'].map(CATEGORY_MAP)
log('category', 'normalized category names and standardized casing/punctuation', len(df))

print("\nAfter normalization:")
print(df['category'].value_counts())

Before normalization:
category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized category names and standardized casing/punctuation (275 row(s))

After normalization:
category
Food        95
RainGear    86
Merch       51
Apparel     43
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()).issubset({'Food', 'RainGear', 'Merch', 'Apparel'})

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

rev_by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
total_revenue = round(df['revenue'].sum(), 2)

print("--- Revenue by Category ---")
print(rev_by_cat)
print("\nTotal Revenue: $", total_revenue)

--- Revenue by Category ---
category
Food        1656.0
RainGear    1512.0
Merch        856.5
Apparel      715.5
Name: revenue, dtype: float64

Total Revenue: $ 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more Food (Cheeseburgers), as it generated the highest total revenue (\$1,656.00 across 95 valid orders), closely followed by RainGear (\$1,512.00).

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,stripped dollar signs and converted price colu...,300
2,qty_filter,dropped rows with missing or negative quantities,25
3,item,canonicalized item spellings into 3 standard p...,275
4,category,normalized category names and standardized cas...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** Dropping exact duplicates (Step 1) changed total revenue the most, reducing it from **\$4,852.50** (raw dataset with duplicate rows) to **\$4,594.50** (after dropping 15 exact duplicate rows), a net change of **-\$258.00**.

**b) Category Consolidation (Apparel vs Merch):** I decided to keep `Apparel` (\$715.50) and `Merch` (\$856.50) as two distinct categories. A reasonable person could have consolidated `Apparel` into `Merch` to combine all non-gear merchandise. Choosing to merge them would combine their totals into a single `Merch` category total of **\$1,572.00**, making `Merch` the second-highest revenue category instead of third and fourth. I kept them separate to retain item-level visibility between wearable apparel and novelty accessories.